In [1]:
%pip install torch torchvision --upgrade --no-cache-dir


Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126 --upgrade --no-cache-dir


Looking in indexes: https://download.pytorch.org/whl/cu126
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
# 1. Converts data types: It turns raw image files into PyTorch Tensors that your GPU can actually understand.
# 2. Standardizes math: It shrinks pixel values from 0 to 255 down to a balanced 
#    range like -1 to 1 so the model learns much faster.

In [4]:
#image=>scaler(0,1)=>normalize=>(-1,1)

In [6]:
# Chaining operations: Compose bundles multiple image transformation steps 
# into a single pipeline that executes sequentially on every image.
transform = transforms.Compose([
    
    # STEP 1: Crucial conversion step. It takes a raw PIL image or NumPy array 
    # (with shape HxWxC and values 0-255) and converts it into a PyTorch Tensor 
    # (with shape CxHxW), while automatically scaling all pixel values down to [0.0, 1.0].
    transforms.ToTensor(),
    
    # STEP 2: Standardizes the image data. Using the formula (pixel - mean) / std, 
    # this shifts the pixels from [0, 1] down to a range of [-1, 1] for all 3 color 
    # channels (Red, Green, Blue). This is why there are three 0.5 values in each tuple.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [9]:
# Loads the training data (50,000 images) to teach the model
trainset = CIFAR10(root ="%data" , train = True, download = True, transform = transform)

# Loads the testing data (10,000 images) to evaluate the model
# FIX: Changed 'Frue' to 'False'
testset = CIFAR10(root ="%data" , train = False, download = True, transform = transform)


In [10]:
# Print the exact number of samples in each set
print("Training images:", len(trainset))  # Outputs: 50000
print("Testing images:", len(testset))    # Outputs: 10000

# Inspect the shape of the data
images, labels = trainset[0]
print("Image tensor shape:", images.shape) # Outputs: torch.Size([3, 32, 32])


Training images: 50000
Testing images: 10000
Image tensor shape: torch.Size([3, 32, 32])


In [12]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # BLOCK 1: Takes 3 color channels (RGB) -> outputs 32 feature maps. 
            # Padding=1 keeps spatial size at 32x32 pixels.
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # BLOCK 2: Takes 32 feature maps -> outputs 64 feature maps.
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # BLOCK 3: Takes 64 feature maps -> outputs 128 feature maps.
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(  # Bundles the classification layers (Fully Connected / Dense layers)


            # Linear Layer 1: Flattens the final 3D feature volume (4x4 pixels x 128 channels = 2048 numbers) 
            # and maps it down to 256 hidden features.
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            

            # Linear Layer 2 (Output Layer): Maps the 256 features down to 10 outputs.
            # These 10 outputs are the unnormalized raw scores (logits) representing your 10 CIFAR-10 classes.
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)#pass image through all convolutional blocks
        x = x.view(x.size(0), -1) # flattening tensor
        x = self.fc_layers(x) # pass flatten features through dense layers

        return x

In [17]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [18]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3728515787045363
epoch=2/10 & loss=0.9435331343727953
epoch=3/10 & loss=0.7584075561874662
epoch=4/10 & loss=0.6276473792084037
epoch=5/10 & loss=0.5333497677846333
epoch=6/10 & loss=0.43821935233710063
epoch=7/10 & loss=0.35535936114733174
epoch=8/10 & loss=0.28617398251238685
epoch=9/10 & loss=0.22071200197614976
epoch=10/10 & loss=0.17840741071230767


### Evalute our CNN

In [21]:
correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
    for images , labels in testloader:
        outputs = model.forward(images)
        _ , predicted = torch.max(outputs , 1)
        correct_labels+=(predicted == labels).sum().item()
        total_labels +=labels.size(0)
    print(f"accuracy = {correct_labels/ total_labels*100}")

accuracy = 75.32
